In [ ]:
# Example 1: AgentsClient class is importable and has expected signature.
import inspect

from azure.ai.agents import AgentsClient

print(f"AgentsClient = {AgentsClient}")
print(f"module = {AgentsClient.__module__}")

sig = inspect.signature(AgentsClient.__init__)
params = list(sig.parameters.keys())
print(f"__init__ params = {params}")

# Sync client should accept endpoint + credential
assert 'endpoint' in params, f"expected 'endpoint' param, got {params}"
assert 'credential' in params, f"expected 'credential' param, got {params}"
assert callable(AgentsClient)

In [ ]:
# Example 2: Async AgentsClient mirrors the sync surface.
import inspect

from azure.ai.agents.aio import AgentsClient as AsyncAgentsClient

print(f"AsyncAgentsClient = {AsyncAgentsClient}")
print(f"module = {AsyncAgentsClient.__module__}")

sig = inspect.signature(AsyncAgentsClient.__init__)
params = list(sig.parameters.keys())
print(f"async __init__ params = {params}")

assert 'endpoint' in params
assert 'credential' in params
# Sync and async client surface should match on these key params
assert AsyncAgentsClient.__name__ == 'AgentsClient'

In [ ]:
# Example 3: Construct a few model objects from azure.ai.agents.models.
from azure.ai.agents import models

# MessageRole enum should expose USER / AGENT
print(f"MessageRole.USER = {models.MessageRole.USER}")
print(f"MessageRole.AGENT = {models.MessageRole.AGENT}")

# FunctionTool / CodeInterpreterTool definitions are present
ci_tool = models.CodeInterpreterTool()
print(f"CodeInterpreterTool = {ci_tool}")
print(f"definitions = {ci_tool.definitions}")

assert models.MessageRole.USER == "user"
assert isinstance(ci_tool.definitions, list)
assert len(ci_tool.definitions) >= 1

In [ ]:
# Example 4: FunctionTool wraps user-defined Python functions into Agent tool specs.
from azure.ai.agents.models import FunctionTool

def get_weather(city: str) -> str:
    """Get the current weather for a city."""
    return f"sunny in {city}"

tool = FunctionTool(functions={get_weather})
defs = tool.definitions
print(f"FunctionTool definitions count = {len(defs)}")
first = defs[0]
print(f"first def type = {type(first).__name__}")
# Each definition should serialize to a dict with name + description
as_dict = first.as_dict() if hasattr(first, 'as_dict') else dict(first)
print(f"keys = {sorted(as_dict.keys())}")
assert len(defs) == 1
assert 'function' in as_dict or 'type' in as_dict

In [ ]:
# Example 5: ToolSet aggregates multiple tools and exposes a unified definitions list.
from azure.ai.agents.models import CodeInterpreterTool, FunctionTool, ToolSet

def add(a: int, b: int) -> int:
    """Add two numbers."""
    return a + b

toolset = ToolSet()
toolset.add(FunctionTool(functions={add}))
toolset.add(CodeInterpreterTool())

all_defs = toolset.definitions
print(f"ToolSet total definitions = {len(all_defs)}")
for d in all_defs:
    print(f"  - {type(d).__name__}")

assert len(all_defs) >= 2, f"expected >=2 defs, got {len(all_defs)}"
print("\nazure_ai_agents smoke test passed")